In [1]:
# For handling the timeseries
import pandas as pd, os, datetime
import numpy as np

# Statistical analysis
from scipy import stats
from scipy.stats import mannwhitneyu
import math

# For plotting
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import plotly.graph_objects as go
import plotly.io as pio
import plotly.express as px
import matplotlib.lines as mlines
import cartopy.feature as cfeature
import cartopy.crs as ccrs
import seaborn as sns
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
pio.renderers.default = 'notebook'

# Import the loader function
os.chdir('/g/data/ng72/ms5578/ID_HW_BARRA')
from process_code import load_generation_data

/g/data/xp65/public/apps/med_conda/envs/analysis3-25.10/lib/python3.11/site-packages/distributed/diagnostics/nvml.py:14: FutureWarning:

The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.

/g/data/xp65/public/apps/med_conda/envs/analysis3-25.10/lib/python3.11/site-packages/distributed/node.py:187: UserWarning:

Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 37309 instead

/g/data/xp65/public/apps/med_conda/envs/analysis3-25.10/lib/python3.11/site-packages/distributed/diagnostics/nvml.py:14: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml
/g/data/xp65/public/apps/med_conda/envs/analysis3-25.10/lib/python3.11/site-

Dask dashboard: /proxy/37309/status


2025-11-21 14:08:04,098 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle c277e47be041ff2cfd925b09a29ecef6 initialized by task ('shuffle-transfer-c277e47be041ff2cfd925b09a29ecef6', 0) executed on worker tcp://127.0.0.1:38999
2025-11-21 14:08:08,537 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle c277e47be041ff2cfd925b09a29ecef6 deactivated due to stimulus 'task-finished-1763694488.489159'
2025-11-21 14:08:09,985 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle d1ad18087374590d565308492fa85d86 initialized by task ('shuffle-transfer-d1ad18087374590d565308492fa85d86', 0) executed on worker tcp://127.0.0.1:33681
2025-11-21 14:08:12,697 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle d1ad18087374590d565308492fa85d86 deactivated due to stimulus 'task-finished-1763694492.6127002'


In [2]:
df, info = load_generation_data(
    sdate="2009-07-01",
    edate="2024-06-30",
    mode="hourly",
    ftype=["Solar"],
    apply_remove_negatives=True,
    apply_remove_wind_zeros=False,
    apply_min_heatwave_days=True,
    min_heatwave_days_threshold=20,
    apply_clear_agc=False
)

Read gen_details & hw_tseries with Dask: 0.27 sec
Select group: 0.00 sec
--- Starting Dask-Native Process ---


/g/data/ng72/ms5578/ID_HW_BARRA/process_code.py:168: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



Starting final Dask compute...
Dask compute finished.

--- DASK TIMING REPORT ---
duid_setup: 0.00 seconds
csv_bulk_read: 0.24 seconds
filter_and_clean: 0.01 seconds
type_conversion_and_dropna: 0.02 seconds
date_filter: 0.01 seconds
hw_tseries_filter: 8.89 seconds
aggregate_hourly: 0.02 seconds
jitter: 0.22 seconds
final_merge: 0.02 seconds
compute: 66.96 seconds
--- END REPORT ---

Process group: 80.11 sec

--- DEBUG: Calculated Heatwave Days per Generator ---
DUID
MOREESF1    271
NYNGAN1     210
KSP1        186
WRSF1       164
SMCSF1      164
BROKENH1    162
PARSF1      133
DDSF1       127
MANSLR1     120
RRSF1       120
CLARESF1    109
SRSF1       109
CSPVPS1     105
EMERASF1    104
CHILDSF1    104
RUGBYR1     102
WHITSF1     101
HAMISF1     101
DAYDSF1     101
HAYMSF1     101
CLERMSF1     96
LILYSF1      96
GULLRSF1     91
COLEASF1     89
NEVERSF1     87
OAKEY1SF     87
OAKEY2SF     84
BERYLSF1     77
KARSF1       75
BANN1        69
GANNSF1      68
WEMENSF1     65
BNGSF2       57
B

In [32]:
df2 = df[df['EHF_flag']==0]
hour_df = df[['DUID','time','TOTALMWh']].copy()
hour_df['hour'] = df['time'].dt.hour
hour_df['date'] = df['time'].dt.date
hour_df = pd.pivot_table(hour_df, index=df[['DUID','date']], columns='hour', values='TOTALMWh')
hour_df = hour_df.dropna()

In [33]:
hour_df

hour                0    1    2    3    4    5    6         7          8   \
DUID   date                                                                 
BANN1  2018-08-01  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.000000   0.000000   
       2018-08-02  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.000000   0.000000   
       2018-08-03  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.000000   0.000000   
       2018-08-04  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.000000   0.000000   
       2018-08-05  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.000000   0.000000   
...                ...  ...  ...  ...  ...  ...  ...       ...        ...   
YATSF1 2024-06-25  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.151667   6.972500   
       2024-06-26  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.064167  10.190833   
       2024-06-27  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.063333  10.132500   
       2024-06-28  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.072500   8.700000   
       2024-06-29  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.000000   2.545833   

hour                      9   ...         14         15         16      17  \
DUID   date                   ...                                            
BANN1  2018-08-01   0.000000  ...   0.000000   0.000000   0.000000  0.0000   
       2018-08-02   0.000000  ...   0.000000   0.000000   0.000000  0.0000   
       2018-08-03   0.000000  ...   0.000000   0.000000   0.000000  0.0000   
       2018-08-04   0.000000  ...   0.000000   0.000000   0.000000  0.0000   
       2018-08-05   0.000000  ...   0.000000   0.000000   0.000000  0.0000   
...                      ...  ...        ...        ...        ...     ...   
YATSF1 2024-06-25  24.906667  ...  19.056667  20.355833  20.178333  0.5125   
       2024-06-26  41.200833  ...  37.199167  45.052500  16.104167  1.2625   
       2024-06-27  40.841667  ...  43.329167  44.779167  25.350000  1.1850   
       2024-06-28   7.498333  ...  18.510000  28.375000   3.861667  0.3775   
       2024-06-29   2.117500  ...   4.016667   1.215000   1.548333  0.0000   

hour                18   19   20   21   22   23  
DUID   date                                      
BANN1  2018-08-01  0.0  0.0  0.0  0.0  0.0  0.0  
       2018-08-02  0.0  0.0  0.0  0.0  0.0  0.0  
       2018-08-03  0.0  0.0  0.0  0.0  0.0  0.0  
       2018-08-04  0.0  0.0  0.0  0.0  0.0  0.0  
       2018-08-05  0.0  0.0  0.0  0.0  0.0  0.0  
...                ...  ...  ...  ...  ...  ...  
YATSF1 2024-06-25  0.0  0.0  0.0  0.0  0.0  0.0  
       2024-06-26  0.0  0.0  0.0  0.0  0.0  0.0  
       2024-06-27  0.0  0.0  0.0  0.0  0.0  0.0  
       2024-06-28  0.0  0.0  0.0  0.0  0.0  0.0  
       2024-06-29  0.0  0.0  0.0  0.0  0.0  0.0  

[118728 rows x 24 columns]

In [34]:
def df_autocorr(df, lag=1, axis=0):
    """Compute full-sample column-wise autocorrelation for a DataFrame."""
    return df.apply(lambda col: col.autocorr(lag), axis=axis)
    
df_autocorr(hour_df)

hour
0     0.291945
1     0.274433
2     0.278528
3     0.274745
4     0.127793
5     0.737582
6     0.783755
7     0.761489
8     0.731828
9     0.723538
10    0.730026
11    0.756135
12    0.763054
13    0.741638
14    0.715572
15    0.692705
16    0.707644
17    0.753897
18    0.765999
19    0.752056
20    0.280805
21    0.358864
22    0.364836
23    0.366614
dtype: float64